# 03 - ESM-2 Embedding Extraction

This notebook preserves the original ESM-2 pooled embedding extraction and integrates the newer per-residue embedding generation required for rare-mutation-aware attention.

GPU notebook: strongly recommended in Google Colab.


In [ ]:
# Colab/local project setup — see COLAB_SETUP.md
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
_notebooks = _nb if (_nb / 'notebook_setup.py').exists() else _nb / 'notebooks'
if str(_notebooks) not in sys.path:
    sys.path.insert(0, str(_notebooks))

from notebook_setup import bootstrap_notebook_environment, export_notebook_globals

paths, pipeline_config = bootstrap_notebook_environment(
    install_deps=True,
    mount_drive=True,
)
globals().update(export_notebook_globals(paths, pipeline_config))

print('Project root:', PROJECT_ROOT)
print('Running in Colab:', IN_COLAB)
print('ENABLE_IMPROVED_PIPELINE:', ENABLE_IMPROVED_PIPELINE)



---

## Original HIV-ESM-2 Embedding Workflow

The cells below are preserved from the original embedding extraction notebook.


# Notebook 03: ESM-2 Embedding Extraction

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Extract protein language model embeddings from ESM-2.

**Model**: esm2_t33_650M_UR50D (650M parameters, 1280-dim embeddings)

**Approach**:
- Extract token-level embeddings from final layer
- Compare pooling strategies (mean, max, attention-weighted)

**Hardware**: GPU with 16+ GB VRAM recommended

---

In [ ]:
# Install ESM-2 if needed
# !pip install -q fair-esm torch

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_processing import load_fasta, load_unified_data
from src.feature_engineering import (
    load_esm2_model, extract_embeddings,
    batch_extract_pooled_embeddings,
    mean_pooling, max_pooling, mean_max_pooling
)

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
EMBEDDINGS_DIR = PROJECT_ROOT / 'data' / 'embeddings'

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Embeddings will be saved to: {EMBEDDINGS_DIR}")

## 1. Load Data

In [ ]:
# Load unified data
unified_data = load_unified_data(DATA_DIR)

print("\nLoaded data:")
for dc, data in unified_data.items():
    n_seqs = len(data['sequences'])
    seq_lengths = [len(s) for s in data['sequences']]
    print(f"  {dc}: {n_seqs} sequences ({np.mean(seq_lengths):.0f} aa avg)")

## 2. Load ESM-2 Model

In [ ]:
model = alphabet = batch_converter = device = None
def ensure_esm2_model_loaded():
    global model, alphabet, batch_converter, device
    if model is None:
        print('Loading ESM-2 model...')
        model, alphabet, batch_converter, device = load_esm2_model(model_name='esm2_t33_650M_UR50D')
        print(f'Model loaded on: {device}; layers: {model.num_layers}; dimension: 1280')
print('ESM-2 model loading is deferred until a cache miss is found.')


## 3. Extract Embeddings

Extract embeddings with different pooling strategies:
1. Mean pooling (average across positions)
2. Max pooling (maximum across positions)
3. Mean + Max concatenation

In [ ]:
# Configuration
BATCH_SIZE = 4  # Reduce if running out of GPU memory
POOLING_METHODS = ['mean', 'max', 'mean_max']

In [ ]:
# Extract embeddings for each drug class
for drug_class, data in unified_data.items():
    print(f"\n{'='*60}")
    print(f"EXTRACTING {drug_class} EMBEDDINGS")
    print(f"{'='*60}")
    
    sequences = data['sequences']
    n_seqs = len(sequences)
    
    for pooling_method in POOLING_METHODS:
        output_path = EMBEDDINGS_DIR / f'{drug_class}_pooled_{pooling_method}.npy'
        
        cache_ok = False
        if output_path.exists():
            try:
                cached = np.load(output_path, mmap_mode='r', allow_pickle=False)
                expected_dim = 2560 if pooling_method == 'mean_max' else 1280
                cache_ok = cached.shape == (n_seqs, expected_dim)
                if cache_ok:
                    print(f"\n{pooling_method}: Reusing valid cache {cached.shape}")
            except Exception as exc:
                print(f"\n{pooling_method}: Invalid cache ({exc}); replacing it after extraction")
        if cache_ok:
            continue
        ensure_esm2_model_loaded()
        print(f"\n{pooling_method}: Extracting...")
        embeddings = batch_extract_pooled_embeddings(
            sequences,
            model, alphabet, batch_converter, device,
            pooling_method=pooling_method,
            batch_size=BATCH_SIZE,
            repr_layer=33
        )
        
        # Save embeddings
        temp_path = output_path.with_name(output_path.name + '.tmp')
        with temp_path.open('wb') as stream:
            np.save(stream, embeddings)
        temp_path.replace(output_path)
        size_mb = embeddings.nbytes / 1e6
        print(f"  Shape: {embeddings.shape}")
        print(f"  Saved to: {output_path.name} ({size_mb:.1f} MB)")

## 4. Verify Embeddings

In [ ]:
print("\nVerifying saved embeddings:")
print("-" * 50)

for drug_class in unified_data.keys():
    print(f"\n{drug_class}:")
    
    for pooling_method in POOLING_METHODS:
        filepath = EMBEDDINGS_DIR / f'{drug_class}_pooled_{pooling_method}.npy'
        
        if filepath.exists():
            embeddings = np.load(filepath)
            size_mb = filepath.stat().st_size / 1e6
            
            # Check for NaN/Inf
            has_nan = np.isnan(embeddings).any()
            has_inf = np.isinf(embeddings).any()
            
            status = "OK" if not (has_nan or has_inf) else "ISSUE"
            print(f"  {pooling_method}: {embeddings.shape} - {size_mb:.1f} MB - {status}")
        else:
            print(f"  {pooling_method}: NOT FOUND")

## 5. Embedding Statistics

In [ ]:
import matplotlib.pyplot as plt

# Load mean pooled embeddings and visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, drug_class in enumerate(unified_data.keys()):
    ax = axes[idx]
    
    embeddings = np.load(EMBEDDINGS_DIR / f'{drug_class}_pooled_mean.npy')
    
    # Plot embedding norm distribution
    norms = np.linalg.norm(embeddings, axis=1)
    ax.hist(norms, bins=50, edgecolor='white', alpha=0.7)
    ax.set_xlabel('Embedding L2 Norm')
    ax.set_ylabel('Count')
    ax.set_title(f'{drug_class}\nMean: {norms.mean():.2f} +/- {norms.std():.2f}')

plt.tight_layout()
norms_path = EMBEDDINGS_DIR / 'embedding_norms.png'
if not norms_path.exists():
    plt.savefig(norms_path, dpi=150, bbox_inches='tight')
else:
    print(f'Reusing existing figure: {norms_path}')
plt.show()

In [ ]:
# Dimensionality reduction visualization (PCA)
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, drug_class in enumerate(unified_data.keys()):
    ax = axes[idx]
    
    embeddings = np.load(EMBEDDINGS_DIR / f'{drug_class}_pooled_mean.npy')
    
    # PCA
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(embeddings)
    
    ax.scatter(pca_coords[:, 0], pca_coords[:, 1], alpha=0.5, s=10)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{drug_class} Embeddings (PCA)')

plt.tight_layout()
pca_path = EMBEDDINGS_DIR / 'embedding_pca.png'
if not pca_path.exists():
    plt.savefig(pca_path, dpi=150, bbox_inches='tight')
else:
    print(f'Reusing existing figure: {pca_path}')
plt.show()

## Summary

**Embeddings extracted:**
- Dimension: 1280 (mean/max) or 2560 (mean_max)
- Pooling methods: mean, max, mean_max
- Stored as .npy files in data/embeddings/

**Total size estimate:**
- ~6,308 sequences x 1280 dims x 4 bytes = ~32 MB per pooling method
- ~100 MB total for all pooled embeddings

**Next steps:**
- Notebook 04: Train classifiers and compare to baseline


---

## Per-Residue Embeddings for Enhanced Training

Run this optional GPU section when using the attention model or Option B validation.


In [ ]:
RUN_PER_RESIDUE_EXTRACTION = pipeline_config.RUN_PER_RESIDUE_EXTRACTION
BATCH_SIZE_PER_RESIDUE = pipeline_config.BATCH_SIZE_PER_RESIDUE
STATISTICAL_VALIDATION_SUBSET_SIZE = pipeline_config.IMPROVED_SUBSET_SIZE

if RUN_PER_RESIDUE_EXTRACTION:
    import numpy as np
    from src.data_processing import load_unified_data
    from src.feature_engineering import batch_extract_per_residue_embeddings
    def valid_per_residue(path, expected_rows):
        if not path.exists():
            return None
        try:
            arr = np.load(path, mmap_mode='r', allow_pickle=True)
            if arr.ndim == 3 and arr.shape[0] == expected_rows and arr.shape[-1] == 1280:
                return arr
        except Exception as exc:
            print(f'Ignoring invalid cache {path}: {exc}')
        return None
    def save_atomic(path, values, allow_pickle=False):
        temp_path = path.with_name(path.name + '.tmp')
        with temp_path.open('wb') as stream:
            np.save(stream, values, allow_pickle=allow_pickle)
        temp_path.replace(path)
    unified_data = load_unified_data(PROCESSED_DIR)
    for drug_class, data in unified_data.items():
        sequences = data['sequences']
        class_dir = EMBEDDINGS_DIR / drug_class
        class_dir.mkdir(parents=True, exist_ok=True)
        root_full = EMBEDDINGS_DIR / f'{drug_class}_per_residue.npy'
        class_full = class_dir / 'per_residue.npy'
        root_array = valid_per_residue(root_full, len(sequences))
        class_array = valid_per_residue(class_full, len(sequences))
        full = root_array if root_array is not None else class_array
        if full is None:
            ensure_esm2_model_loaded()
            print(f'Extracting missing per-residue embeddings for {drug_class} (n={len(sequences)})')
            per_residue = batch_extract_per_residue_embeddings(sequences, model, alphabet, batch_converter, device, batch_size=BATCH_SIZE_PER_RESIDUE, repr_layer=33)
            per_residue = np.asarray(per_residue, dtype=object)
            save_atomic(root_full, per_residue, allow_pickle=True)
            save_atomic(class_full, per_residue, allow_pickle=True)
            full = valid_per_residue(root_full, len(sequences))
            if full is None:
                raise RuntimeError(f'New per-residue array failed validation: {root_full}')
        else:
            print(f'Reusing valid full per-residue embeddings for {drug_class}: {full.shape}')
            if root_array is None:
                save_atomic(root_full, full)
            if class_array is None:
                save_atomic(class_full, full)
        subset_path = EMBEDDINGS_DIR / f'{drug_class}_per_residue_sub_{STATISTICAL_VALIDATION_SUBSET_SIZE}.npy'
        subset_rows = min(STATISTICAL_VALIDATION_SUBSET_SIZE, len(sequences))
        subset = valid_per_residue(subset_path, subset_rows)
        if subset is None:
            print(f'Rebuilding invalid/missing subset from saved full embeddings: {subset_path.name}')
            save_atomic(subset_path, np.asarray(full[:subset_rows]))
        else:
            print(f'Reusing valid subset embeddings: {subset_path.name} {subset.shape}')
else:
    print('Set RUN_PER_RESIDUE_EXTRACTION = True to validate/reuse per-residue caches.')


## Workflow Handoff

Next notebook: `04_baseline_and_enhanced_training.ipynb`.

Per-residue embeddings (`*_per_residue.npy`) are required for attention pooling and notebook 07 improved pipeline when `ENABLE_IMPROVED_PIPELINE = True`.
